# 5 · AI Agents for Finance

**Outcome of this session:** a working investment-analyst agent, together with your capstone presentation.

**In this notebook you will:**

- Read a complete agent loop and locate every governance lever in it
- Build the tool and the rules that protect an agent from a currency trap
- Run a governed agent on a company pair of your choice
- Meet PydanticAI and LangGraph, and map them to the loop you already understand


## Workflow vs agent: the distinction that matters

| | Workflow (yesterday) | Agent (today) |
|---|---|---|
| Plan | fixed, written by you | **chosen by the model**, step by step |
| Tools | called by your code | **requested** by the model, executed by your code |
| Stops when | the script ends | the model judges the goal met: or hits YOUR limits |
| Failure mode | a step errors loudly | wanders, loops, or is confidently wrong |

**An agent is a loop.** Model proposes a tool call → your code executes it → the result goes back → repeat. Governance is not a policy document; it's these levers *in the code*: `max_steps`, a token budget, a tool whitelist, forced structured output, a human gate, and an audit trail.

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A: anatomy of the agent
We import the course agent and inspect its specification (rules, tools, limits) before granting it any autonomy. Review the specification, not only the output.

In [ ]:
sys.path.insert(0, str(ROOT / "session-05-agents" / "demo"))
import mini_analyst_agent as agent
from toolkit import llm            # for llm.show(): renders long output readably

print(agent.SYSTEM)
print("TOOLS:", [t["name"] for t in agent.TOOLS])
print(f"Guardrails: max_steps={agent.MAX_STEPS}, token_budget={agent.TOKEN_BUDGET:,}")

Read `run_agent` in `session-05-agents/demo/mini_analyst_agent.py` once, carefully. It is roughly forty lines, and it contains the entire concept. Note where each guardrail lives, and that tool errors are sent BACK to the model (it can adapt), while a malformed recommendation is REJECTED and returned for correction (validation at the boundary: notebook 04's lesson, applied to the agent's own output).

## Part B: LAB: build the agent's hands and rules

### Exercise 1: the comparison tool with the currency trap

Novo Nordisk files in **Danish kroner**. A naive agent compares 309bn DKK to 65bn USD and calls Novo "5× bigger". Your tool must detect differing units, add a LOUD `warning` key for the model, and `print()` it so the human watching the trace sees the trap being caught.

In [ ]:
def tool_compare_metrics(args: dict) -> str:
    """Growth and margins side-by-side, computed in code. args = {"tickers": [...]}."""
    rows = [agent._metrics_row(t) for t in args["tickers"]]
### START CODE HERE ###
    units = {r[None] for r in rows}                       # which field differs across foreign filers?
    out = {"companies": rows}
    if len(units) > None:                                 # warn when how many distinct units?
        out["warning"] = (f"UNITS DIFFER ({', '.join(sorted(units))}): absolute amounts are NOT "
                          "comparable. Compare growth and margins only, or convert currency first.")
        print(f"⚠️ compare_metrics: UNITS DIFFER ({', '.join(sorted(units))}) - flagged to the agent")
### END CODE HERE ###
    return json.dumps(out, indent=1)

_ = tool_compare_metrics({"tickers": ["NVO", "LLY"]})

In [ ]:
# ✅ self-check: run me (live EDGAR, no API key needed)
out = json.loads(tool_compare_metrics({"tickers": ["NVO", "LLY"]}))
assert "warning" in out, "NVO reports in DKK, LLY in USD - your tool must add the warning key"
assert "DKK" in out["warning"] and "USD" in out["warning"]
same = json.loads(tool_compare_metrics({"tickers": ["LLY", "MRK"]}))
assert "warning" not in same, "same-currency pairs should NOT warn"
print("All checks passed ✅ - the trap is armed")

### Exercise 2: the agent's operating rules

Write the system prompt. It must cover:

- plan first
- gather every company before comparing
- **always check the `unit` field; never compare absolute amounts across currencies**
- use `compare_metrics` for arithmetic
- every figure comes from tool results only
- `insufficient_data` is an acceptable stance
- finish with `record_recommendation`, exactly once
- this is coursework, not investment advice

In [ ]:
### START CODE HERE ###
MY_SYSTEM = """You are an investment research agent for an educational exercise.

Operating rules:
1. PLAN first: say briefly which tools you will call and why.
2. Gather financials for EVERY company involved before comparing anything.
3. ALWAYS check the '[WHICH FIELD?]' field. Never compare absolute amounts across
   different [ACROSS WHAT?] - compare unitless measures (growth, margins) and say so.
4. Use [WHICH TOOL?] for arithmetic. Do not compute numbers yourself.
5. Every figure you state must come from a tool result in this conversation.
6. If data is missing or ambiguous, use stance '[WHICH STANCE?]'.
7. Finish by calling record_recommendation EXACTLY ONCE. This is coursework,
   not investment advice.
"""
### END CODE HERE ###
print(MY_SYSTEM)

In [ ]:
# ✅ self-check: run me
low = MY_SYSTEM.lower()
for needle, hint in [("unit", "rule about checking the unit field"),
                     ("curren", "forbid cross-currency absolute comparisons"),
                     ("compare_metrics", "arithmetic goes through the tool"),
                     ("record_recommendation", "must finish with the structured recommendation"),
                     ("insufficient", "allow an insufficient_data stance"),
                     ("not investment advice", "coursework disclaimer")]:
    assert needle in low, f"missing: {hint}"
print("All checks passed ✅")

### Exercise 3: run your agent

This exercise calls the API (a run costs a few cents), so it needs your key configured.

We connect your rules and your tool to the course loop and run it on a company pair. Follow the trace: the plan, the tool calls, the moment the currency check fires, and the structured recommendation.

In [ ]:
if HAS_KEY:
### START CODE HERE ###
    agent.SYSTEM = None                               # wire in YOUR rules from Exercise 2
    agent.TOOL_IMPLS["compare_metrics"] = None        # wire in YOUR tool from Exercise 1
### END CODE HERE ###

    TASK = ("Analyze Novo Nordisk (ticker NVO) and compare it with Eli Lilly "
            "(ticker LLY): which is better positioned on growth and profitability?")
    # Capstone: swap in YOUR OWN pair of SEC filers (pharma, banking or tech).

    rec, trace = agent.run_agent(TASK)
    if rec:
        llm.show(
            f"**{rec['headline']}**\n\n"
            f"*Stance: {rec['stance']} · Confidence: {rec['confidence']}*\n\n"
            "**Key points**\n\n"
            + "\n".join(f"- {p}" for p in rec["key_points"])
            + "\n\n**Risks**\n\n"
            + "\n".join(f"- {r}" for r in rec["risks"])
            + f"\n\n**What would change this view:** {rec['what_would_change_my_mind']}",
            title="The agent's recommendation")
        print(f"{len(trace)} tool call(s). Token usage: {llm.usage_summary()}")
else:
    print("No API key - run `python session-05-agents/demo/mini_analyst_agent.py --preflight` "
          "in the terminal to study the agent's spec instead.")

In [ ]:
# The HUMAN GATE: nothing is saved until you approve. Read the recommendation
# above. If - and only if - you'd sign it, set APPROVE = True and run.
APPROVE = False

if HAS_KEY and APPROVE and rec:
    OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
    (OUTD / "agent_memo.md").write_text(agent.render_memo(TASK, rec, trace))
    print("Saved outputs/agent_memo.md - with the full audit trail of every tool call.")
elif HAS_KEY:
    print("Not saved. The human gate is a feature, not a formality.")

## From your forty lines to the industry frameworks

You built the loop raw so that no vendor terminology would ever be opaque to you. Now meet the two names you will hear in interviews, and notice that you already know what they do.

**PydanticAI** packages exactly what you just built: an agent loop with typed, validated output and usage limits. The cell below is your Exercise 3 agent again, in about ten lines. Read it and map each piece to your own code: `output_type` is your forced `record_recommendation` schema, `instructions` is your system prompt, `UsageLimits(request_limit=...)` is your `MAX_STEPS`, and the typed `result.output` is your validated recommendation.

In [ ]:
try:
    from pydantic_ai import Agent
except ModuleNotFoundError:          # first run on an env set up before pydantic-ai joined the course
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic-ai-slim[anthropic]"], check=True)
    from pydantic_ai import Agent

from pydantic import BaseModel
from typing import Literal
from pydantic_ai.usage import UsageLimits

class QuickTake(BaseModel):
    stance: Literal["company_a", "company_b", "insufficient_data"]
    rationale: str

if HAS_KEY:
    framework_agent = Agent(
        "anthropic:claude-sonnet-5",
        output_type=QuickTake,
        instructions="You are an equity analyst. Use ONLY the metrics given. "
                     "If units differ, say insufficient_data. Coursework, not investment advice.",
    )
    # Notebooks already run an event loop, so we await the agent instead of
    # calling run_sync (which would raise "event loop is already running").
    result = await framework_agent.run(
        "Company A: revenue growth 34%, net margin 22%. "
        "Company B: revenue growth 4%, net margin 8%. "
        "Which is better positioned on growth and profitability?",
        usage_limits=UsageLimits(request_limit=3),          # the framework's MAX_STEPS
    )
    print("stance   :", result.output.stance)                # typed, validated, like your rec
    llm.show(result.output.rationale, title="PydanticAI, same discipline, ten lines")
else:
    print("No API key - read the cell instead: every argument maps to a lever you built.")

**LangGraph** is the other name you will hear. It models an agent as an explicit graph: nodes are steps, edges are transitions, and a checkpointer persists the state between runs. You do not need to build one to understand it, because every concept maps onto something you wrote today:

| Your forty lines | PydanticAI | LangGraph |
|---|---|---|
| the `while` loop | the `Agent` run | the graph and its edges |
| `messages` list (the conversation is the memory) | `message_history` | the state object + **checkpointer** (persistent memory) |
| forced `record_recommendation` schema | `output_type=Model` | a typed state schema |
| `MAX_STEPS`, `TOKEN_BUDGET` | `UsageLimits` | recursion limits |
| your human gate before saving | deferred tool approval | **`interrupt()`** before a node |
| the audit trail you print | `result.all_messages()` | the checkpointed state history |

Two professional conclusions. First, every row above still has to be *decided* by you, whichever column you build in: the framework implements the levers, but choosing them remains your work. Second, when a vendor demo says "guardrails, memory, human-in-the-loop", you can now ask the precise question: *which limit, stored where, interrupting what?*

## When not to use an agent

For a repeatable task (reconciliation, screens, reports), a **workflow** is the better choice: cheaper, testable, auditable. For open-ended research whose path depends on the data, an agent fits, with the set of controls you just built. If a regulator asks *"why did it do that?"*, you want either the workflow's fixed plan or the agent's audit trail as your answer. Choose the degree of autonomy per task; it is a setting you control, not a property of the technology.

## Capstone: "My AI-Native Finance Workflow"

Five minutes per person, hard stop.

Full brief + rubric: `session-05-agents/capstone/capstone-brief.md`. The skeleton: **the job** (30s) → **a live run** on your own company pair (2m) → **the trust story** (1m30: one failure you caught + the check that caught it + what stays human) → **one number** (30s) → **next step** (30s). The trust story carries the most weight in grading.

## Deliverable checklist

- [ ] Both ✅ checks green; your agent ran end-to-end on a pair YOU chose
- [ ] The DKK/USD warning appeared in a trace at least once
- [ ] You can point to every guardrail in the code (max_steps, budget, whitelist, forced output, gate, audit trail)
- [ ] `outputs/agent_memo.md` saved through the human gate and committed
- [ ] Capstone story drafted

**You now own the whole stack:** playbook → comparable-company analysis tool → tests & evidence verification → EDGAR workflows → a governed agent. *Never ship a number you haven't verified.* A good next step: choose one recurring task at your desk, build it as a workflow, and show a colleague.